In [0]:
%py
print('heloo')

In [0]:
%sql
SHOW TABLES IN formula1_catalog.bronze

In [0]:
%run ../00-common/01.environment-config

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.constructors"
silver_table=f"{catalog_name}.{silver_schema}.constructors"


In [0]:

bronze_table

In [0]:
%sql
describe history formula1_catalog.bronze.constructors

In [0]:
# spark.read for aditonal options to read table data
#ciucuits_df=spark.read.option('versionAsOf',0).table(bronze_table)

In [0]:
constructors_df=spark.table(bronze_table)

In [0]:
constructors_df_selected=constructors_df.drop("url")

In [0]:
# from pyspark.sql import functions as F
# constructors_df_selected=constructors_df.select(
#     F.col("constructorId"),
#     F.col("name"),
#     F.col("nationality"),
#     F.col("ingestion_timestamp"),
#     F.col("source_file"))


In [0]:
constructors_renamed_df=(
    constructors_df_selected
        .withColumnsRenamed ({
                     "constructorId":"constructor_id",
                     "name":"constructor_name"
                     })  
                    
)

In [0]:
from pyspark.sql import functions as F
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter("circuit_id is not  null")
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter(circuits_renamed_df['circuit_id'].isNotNull())
constructors_renamed_nulldroped_df=constructors_renamed_df.filter(
    F.col('constructor_id').isNotNull()
)

In [0]:
display(constructors_renamed_nulldroped_df.count())
display(constructors_renamed_df.count())


In [0]:
#circuits_distinct_df=circuits_renamed_nulldroped_df.distinct()
constructors_distinct_df=constructors_renamed_nulldroped_df.dropDuplicates(["constructor_id"])
display(constructors_distinct_df)

In [0]:

# duplicates = races_renamed_df[["season", "round"]].groupBy("season", "round").count().filter("count > 1")
# display(duplicates)

In [0]:
from pyspark.sql.functions import initcap
constructors_final_df=(constructors_distinct_df
    .withColumn('nationality',F.initcap(F.col('nationality')))
 )

In [0]:
display(constructors_final_df)

In [0]:
(
    constructors_final_df
        .write
        .format("delta")
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1_catalog.silver.constructors